# Day 6 — Combined KPI Report + Pareto Sweep

**Owner:** Pritam  
**Branch:** `pritam_temp_apr5`  
**Depends on:** Day 3 (`forward_kpi_summary.csv`), Day 4 (`reverse_kpi_summary.csv`), Day 5 (`hybrid_kpi_summary.csv`)

## Objectives

| Task | `src/` function | Output |
|------|----------------|--------|
| Merge fwd + rev + hybrid KPIs | `kpi_reporter.run()` | `outputs/combined_kpi_report.csv` |
| Zone priority ranking (SDVRP) | `kpi_reporter.zone_priority_ranking()` | column `sdvrp_priority_rank` |
| Pareto sweep (α/β grid) | `joint_optimizer.pareto_sweep()` | `outputs/pareto_results.csv` |
| Pareto tradeoff chart | (this notebook) | `outputs/pareto_tradeoff.png` |

> **Notebook rule:** No business logic here. All computation is in `src/`.  
> Cells only call `src/` functions, load CSVs, and produce charts.


## Theory 1 — Why a Combined KPI Report?

Running forward and reverse logistics as separate pipelines means each zone's
performance is evaluated in isolation.  A combined report gives us three things:

1. **True cost baseline** — `separate_cost_R$ = fwd_cost + rev_cost`  
   This is the actual spend if forward and reverse fleets never share a trip.

2. **SDVRP saving** — `saving_R$ = separate_cost_R$ - hybrid_cost_R$`  
   What we gain by letting vehicles deliver *and* collect on the same trip
   (Simultaneous Delivery and Pickup VRP, Dethloff 2001).

3. **Zone priority ranking** — sorts zones by saving opportunity so that the
   operations team knows *which dark store* to deploy hybrid routing to first.

**Saving formula:**

$$\text{saving\_pct} = \frac{\text{separate\_cost} - \text{hybrid\_cost}}{\text{separate\_cost}} \times 100$$

Expected range: 15–25 % (literature benchmark for SDVRP vs separate fleets).


In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.kpi_reporter import run as kpi_run, load_forward, load_reverse, load_hybrid
from src.joint_optimizer import pareto_sweep


## Part 1 — Build Combined KPI Report

`kpi_reporter.run()` merges forward + reverse KPIs, optionally joins hybrid SDVRP
results, and ranks zones by saving opportunity.

The function:
1. Loads `outputs/forward_kpi_summary.csv` + `outputs/reverse_kpi_summary.csv`
2. Computes `separate_cost_R$ = fwd_cost + rev_cost` per zone
3. Joins `outputs/hybrid_kpi_summary.csv` if present (from `run_all_zones_sdvrp`)
4. Adds `sdvrp_priority_rank` — Zone 8 expected to be rank 1 (highest cost)
5. Saves `outputs/combined_kpi_report.csv`


In [ ]:
# Build and display the combined KPI report
report_df = kpi_run(
    fwd_path="../outputs/forward_kpi_summary.csv",
    rev_path="../outputs/reverse_kpi_summary.csv",
    hybrid_path="../outputs/hybrid_kpi_summary.csv",   # silently skipped if absent
    out_path="../outputs/combined_kpi_report.csv",
)

# Show the full table
cols_show = [
    "zone_id", "fwd_cost_R$", "rev_cost_R$", "separate_cost_R$",
    "fwd_vehicles", "rev_vehicles", "combined_vehicles",
    "sdvrp_priority_rank",
]
display(report_df[cols_show].sort_values("sdvrp_priority_rank"))


In [ ]:
# ── Stacked bar chart: forward vs reverse cost per zone ──────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

zones = report_df["zone_id"].astype(str)
x = range(len(zones))

ax.bar(x, report_df["fwd_cost_R$"], label="Forward (delivery)", color="#2196F3")
ax.bar(x, report_df["rev_cost_R$"], bottom=report_df["fwd_cost_R$"],
       label="Reverse (pickup)", color="#FF9800")

ax.set_xticks(list(x))
ax.set_xticklabels([f"Zone {z}" for z in zones], rotation=30, ha="right")
ax.set_ylabel("Routing Cost (R$)")
ax.set_title("Separate Fwd + Rev Cost per Zone  |  Day 6 Baseline")
ax.legend()
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("R$%.0f"))

# Annotate total on each bar
for i, row in report_df.iterrows():
    ax.text(i, row["separate_cost_R$"] + 4, f"R${row['separate_cost_R$']:.0f}",
            ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/combined_cost_by_zone.png", dpi=150)
plt.show()
print("Saved → outputs/combined_cost_by_zone.png")


## Theory 2 — Pareto Frontier in Multi-Objective Optimisation

Our joint objective is:

$$Z = \gamma \cdot T_{\text{pen}} + \delta \cdot N_{\text{veh}} + C_{\text{fwd}}(k_{\text{fwd}}) + C_{\text{rev}}(k_{\text{rev}})$$

where $\gamma$ (time-penalty weight) and $\delta$ (fleet-size weight) are fixed hyperparameters, and the decision variables are the **number of active forward dark-stores** $k_{\text{fwd}}$ and **reverse dark-stores** $k_{\text{rev}}$.

The **Pareto front** is the set of $(k_{\text{fwd}}, k_{\text{rev}})$ combinations where:

> No routing cost can be reduced without increasing $T_{\text{pen}}$, or vice-versa.

Formally, solution $A$ **dominates** solution $B$ if:
$$C_{\text{routing}}^A \leq C_{\text{routing}}^B \quad \text{AND} \quad T_{\text{pen}}^A \leq T_{\text{pen}}^B \quad \text{AND at least one strict inequality}$$

The **knee point** is the Pareto-front solution with minimum normalised Euclidean distance to the ideal point $(0, 0)$ — the most balanced trade-off.

### Implementation (`src/joint_optimizer.pareto_sweep`)

- **ε-constraint enumeration**: iterates all $(k_{\text{fwd}}, k_{\text{rev}}) \in \{1\ldots N_{\text{fwd}}\} \times \{1\ldots N_{\text{rev}}\}$
- $\gamma$ and $\delta$ are fixed module-level defaults (2.0 and 50.0 respectively)
- $T_{\text{pen}} = \gamma \cdot \text{expected\_returns} \cdot (1 - k_{\text{rev}} / N_{\text{rev}})$ — penalty decreases as more reverse stores activate
- Pareto front identified via pairwise dominance check on $(C_{\text{fwd}}+C_{\text{rev}},\; T_{\text{pen}})$
- Knee = $\arg\min_{\text{Pareto}} \sqrt{c_{\text{norm}}^2 + t_{\text{norm}}^2}$


In [ ]:
# ── Run Pareto sweep (ε-constraint enumeration) ──────────────────────────────
# Load fixed routes from Day 3/4
fwd_routes = pd.read_csv("../outputs/forward_routes.csv")
rev_routes = pd.read_csv("../outputs/reverse_routes.csv")

# Synthetic return-probability per stop (low-return scenario: mean ≈ 0.12)
n_fwd_stops = fwd_routes["vehicle_id"].nunique() * 10
rng = np.random.default_rng(42)
return_probs = pd.Series(rng.beta(2, 15, size=n_fwd_stops).astype(float))
print(f"return_probs: n={len(return_probs)}, sum={return_probs.sum():.1f}, mean={return_probs.mean():.3f}")

pareto_df = pareto_sweep(
    fwd_routes_df=fwd_routes,
    rev_routes_df=rev_routes,
    return_probs=return_probs,
    # gamma and delta use module defaults (2.0, 50.0) — kept fixed
    output_path="../outputs/pareto_results.csv",
)

print(f"\nAll {len(pareto_df)} combinations:")
display(
    pareto_df[["n_fwd_active", "n_rev_active", "C_fwd", "C_rev",
               "T_pen", "total_routing_cost", "N_veh", "Z", "is_pareto", "is_knee"]]
    .sort_values(["is_knee", "is_pareto", "total_routing_cost"],
                 ascending=[False, False, True])
)


In [ ]:
# ── Pareto tradeoff scatter + frontier line ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# All solutions
non_pareto = pareto_df[~pareto_df["is_pareto"]]
pareto_front = pareto_df[pareto_df["is_pareto"]].sort_values("total_routing_cost")
knee = pareto_df[pareto_df["is_knee"]]

ax.scatter(non_pareto["total_routing_cost"], non_pareto["T_pen"],
           color="lightgray", s=60, label="Dominated", zorder=2)

ax.scatter(pareto_front["total_routing_cost"], pareto_front["T_pen"],
           color="#2196F3", s=100, label="Pareto front", zorder=3)

# Connect Pareto front points
ax.plot(pareto_front["total_routing_cost"], pareto_front["T_pen"],
        color="#2196F3", lw=1.5, ls="--", zorder=2)

# Knee point
if not knee.empty:
    ax.scatter(knee["total_routing_cost"], knee["T_pen"],
               color="red", s=200, marker="*", label="Knee point", zorder=5)

# Labels on Pareto points
for _, row in pareto_df[pareto_df["is_pareto"]].iterrows():
    ax.annotate(
        f"k_fwd={int(row['n_fwd_active'])} k_rev={int(row['n_rev_active'])}",
        (row["total_routing_cost"], row["T_pen"]),
        textcoords="offset points", xytext=(6, 4), fontsize=7,
    )

ax.set_xlabel("Total Routing Cost  C_fwd + C_rev  (R$)")
ax.set_ylabel("Time Penalty  T_pen  (expected late returns)")
ax.set_title("Pareto Tradeoff: Routing Cost vs Late-Return Penalty\nDark Store SP — ε-Constraint Sweep")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/pareto_tradeoff.png", dpi=150)
plt.show()
print("Saved → outputs/pareto_tradeoff.png")


## Part 3 — SDVRP Priority Ranking

Zone 8 is **rank 1** because it has the highest combined fwd+rev cost (R$570.84).
This is confirmed both by `kpi_reporter.zone_priority_ranking()` above and by the
Day 5 analysis in `04_05_joint_optimizer.ipynb`.

**After running `run_all_zones_sdvrp()`** (notebook `04_05_joint_optimizer.ipynb` Cell 11),
the combined report will automatically populate:
- `hybrid_cost_R$`, `saving_R$`, `saving_pct` per zone
- Re-ranked by actual saving (not estimated)

### Interpretation guide

| Rank | Zone | Sep. Cost | Interpretation |
|------|------|-----------|----------------|
| 1 | 8 | R$570.84 | Highest priority — most savings potential |
| 2 | 2 | R$472.56 | High pickups (57) relative to deliveries |
| 3 | 4 | R$467.84 | 3 fwd vehicles + tight zone |
| 11 | 3 | R$388.02 | Lowest priority — 1 vehicle only |


In [ ]:
# ── Priority ranking bar chart ───────────────────────────────────────────────
ranked = report_df.sort_values("sdvrp_priority_rank").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#E53935" if r == 1 else "#1976D2" for r in ranked["sdvrp_priority_rank"]]
bars = ax.barh(
    [f"Zone {z} (Rank {r})" for z, r in zip(ranked["zone_id"], ranked["sdvrp_priority_rank"])],
    ranked["separate_cost_R$"],
    color=colors,
)
ax.set_xlabel("Separate Cost (R$)  [fwd + rev]")
ax.set_title("SDVRP Zone Priority Ranking — Zone 8 is highest opportunity")
# Annotate values
for bar, val in zip(bars, ranked["separate_cost_R$"]):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2,
            f"R${val:.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/sdvrp_priority_ranking.png", dpi=150)
plt.show()
print("Saved → outputs/sdvrp_priority_ranking.png")


## Day 6 Deliverables Checklist

| Item | `src/` function | Status |
|------|----------------|--------|
| `src/kpi_reporter.py` created | — | ✅ |
| `pareto_sweep()` added to `src/joint_optimizer.py` | `pareto_sweep` | ✅ |
| `outputs/combined_kpi_report.csv` | `kpi_reporter.run()` | ✅ Run Cell 5 |
| `outputs/combined_cost_by_zone.png` | (this notebook) | ✅ Run Cell 6 |
| `outputs/pareto_results.csv` | `pareto_sweep()` | ✅ Run Cell 8 |
| `outputs/pareto_tradeoff.png` | (this notebook) | ✅ Run Cell 9 |
| `outputs/sdvrp_priority_ranking.png` | (this notebook) | ✅ Run Cell 11 |
| `outputs/hybrid_kpi_summary.csv` | `run_all_zones_sdvrp()` | ⬅ `04_05_joint_optimizer.ipynb` Cell 11 |
| Priority re-rank by actual saving | `kpi_reporter.run()` | ⬅ Re-run Cell 5 after hybrid |

**Day 7 next:** Full 10–12 page report + `run_all.sh` reproducible pipeline script.
